##  Step 1: Import Required Libraries

In [24]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from fuzzywuzzy import process
import pickle

## Step 2: Load Dataset

In [25]:
# Load Dataset
data = pd.read_csv('data_files/Training.csv')
data.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


##  Step 3: Preprocessing

What is LabelEncoder?
LabelEncoder is a class from sklearn.preprocessing that converts categorical labels (strings) into integers.
Each unique category is assigned a unique number.

In [26]:
# X: All symptoms columns(features)
# y : only prognosis column

X = data.drop('prognosis', axis = 1)

# This line is converting the target variable prognosis (which contains disease names as strings) into numeric values using Label Encoding.
# ['Diabetes', 'Flu', 'Cancer'] → [0, 1, 2]
y = LabelEncoder().fit_transform(data['prognosis'])


## Step 4: Train-Test Split

In [27]:
# Splits the dataset:

# 70% for training (X_train, y_train)
# 30% for testing (X_test, y_test)

# random_state=42: Makes the split reproducible (same every time)


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)


## Step 5: Train Model

#### What is a Random Forest Model?
🌳 A Random Forest is:
An ensemble model made of many Decision Trees (a forest of trees).

Each tree makes a prediction, and the forest votes (majority wins).

🔧 How it works:
Randomly pick subsets of the data.
Train many decision trees independently.
When predicting, each tree votes.
The most common class is the final prediction.


#### Why RandomForestClassifier is used
🔍 Reason:
RandomForestClassifier is a powerful, easy-to-use, and accurate machine learning model—especially when:

You have structured/tabular data (like medical symptoms)
You want high accuracy without too much tuning
Your data has many features (symptoms columns)
You want automatic feature selection (it figures out what's important)


In [28]:
# Explanation:

# RandomForestClassifier: Uses 100 decision trees to predict the disease.
# .fit(): Trains the model on training data.



model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train,y_train)



RandomForestClassifier(random_state=42)

## Step 6: Evaluate Model

In [29]:
prediction = model.predict(X_test)

print("Model Accuracy: ",accuracy_score(y_test,prediction))


Model Accuracy:  1.0


## Step 7: Save Model


In [30]:
pickle.dump(model, open('disease_model.pkl','wb'))

## Step 8: Load Supporting CSVs

In [31]:
description = pd.read_csv('data_files/description.csv')
precautions = pd.read_csv('data_files/precautions_df.csv')
medications = pd.read_csv('data_files/medications.csv')
diets = pd.read_csv('data_files/diets.csv')
workout = pd.read_csv('data_files/workout_df.csv')



# description: Text explaining what the disease is.
# precautions: Things the patient should follow.
# medications: Common medicines.
# diets: Suggested diet for that disease.
# workout: Suggested physical activities.

## Step 9: Symptom & Disease Mapping

In [32]:
# It gets a list of all symptom names (feature names) from your dataset X.
# symptom_index = ['itching', 'skin_rash', 'headache', 'fatigue']

symptom_index = X.columns.tolist()




# This line creates a dictionary that maps each symptom (in a readable format) to its column index in X.
# for i, symptom in enumerate(symptom_index) == Loops through the list of symptoms and gives you both the index i and the name symptom.
# symptom.replace('_', ' ')	== Replaces underscores with spaces for better readability (e.g., 'skin_rash' → 'skin rash')
# lower() ==	Converts symptom names to lowercase (to make matching easier and case-insensitive)
# {...: i}	Builds a dictionary where key = symptom name in nice format, value = its index

symptom_map = {symptom.replace('_',' ').lower():i
                for i, symptom in enumerate(symptom_index)
               }


# ['itching', 'skin_rash', 'headache', 'fatigue']

# symptom_map = {
#   'itching': 0,
#   'skin rash': 1,
#   'headache': 2,
#   'fatigue': 3
# }
# Now, if a user types "skin rash" in a chatbot or form, you can quickly find its index with:


# symptom_map['skin rash']   # Output: 1


## Step 10: Reverse Label Encoding

In [33]:
# It collects all unique disease names (prognoses) from the dataset and stores them as a Python list.

# 🔍 Explanation:
# data['prognosis'] – selects the "prognosis" column (diseases).
# .unique() – finds all the distinct disease names (e.g., "Flu", "Allergy", "Malaria", etc.).
# .tolist() – converts the result from a NumPy array to a Python list.

# ['Flu', 'Cold', 'Malaria']

disease_classes = data['prognosis'].unique().tolist()

# Convert string into numerical values 
label_encoder = LabelEncoder()

label_encoder.fit(data['prognosis'])

# After filttering
# {'Cold': 0, 'Flu': 1, 'Malaria': 2}



LabelEncoder()

## Step 11: Fuzzy Matching for Symptoms

In [34]:
def correct_symptom(symptom):

#  Key parts:
# symptom.lower()
# ➝ Converts the input to lowercase (e.g., "Feverr" → "feverr") so the match is case-insensitive.

# symptom_map.keys()
# ➝ This gives a list of all valid symptoms from your dataset (like "fever", "headache", etc.).

# process.extractOne(...)
# ➝ Comes from the fuzzywuzzy or rapidfuzz library.
# It compares your input ("feverr") with all known symptoms and returns:

# This line uses fuzzy string matching to find the closest match for the symptom the user typed.
    match, score = process.extractOne(symptom.lower(), symptom_map.keys())

# ('fever', 95)
# 'fever' is the closest matching symptom from your data
# 95 is the similarity score (out of 100)

    return match if score >= 80 else None



#  Real-World Example:

# symptom_map = {
#     'headache': 0,
#     'fever': 1,
#     'sore throat': 2,
#     'nausea': 3,
# 
# User types:

# correct_symptom("feverr")  ➝ 'fever'
# correct_symptom("soree throte")  ➝ 'sore throat'
# correct_symptom("xyz123")  ➝ None  (because no close match found)


##  Step 12: Predict Disease

In [41]:
# This function predicts the disease based on list of user-input symptoms.
def predict_disease(symptoms):


# symptom_map contains all known symptoms with their index in the model input.
# We create a list of 0s of length equal to the number of symptoms.
# This list represents the symptom presence vector (1 = present, 0 = absent).

# For example, if there are 5 symptoms:
# [0, 0, 0, 0, 0]  → initially all are set to 0
    input_vector = [0] * len(symptom_map)


    # symptom is a list user-entered symptoms names
    for symptom in symptoms:

        # Try to correct the symptom spelling using fuzzy matching
        corrected = correct_symptom(symptom)
        print(f"Input: {symptom}, Matched: {corrected}")



        #  Set corresponding symptom index to 1
        if corrected:
            # If the symptom is recognized, its position in the input vector is set to 1.
            input_vector[symptom_map[corrected]] = 1

        else:
            print(f"Symptom {symptom} not recongnized")

        
        # Make prediction
    predicated_code = model.predict([input_vector])[0]
        # model is your trained RandomForestClassifier.
        # It predicts the label index (like 5, 10, etc.), representing a disease class.


        # : Convert prediction code back to disease name
    return label_encoder.inverse_transform([predicated_code])[0]
    
    








        


## Step 13: Show Recommendation

In [51]:
import ast



def get_recommendation(disease):

    # Filters the description DataFrame for the row where 'Disease' matches the input disease.
     # Extracts the first matching description text (from the 'Description' column).

    # Check if disease exists in description DataFrame
    if disease not in description['Disease'].values:
        print(f"\n No description found for: {disease}")
        return
    
    # Check if disease exists in precautions DataFrame
    if disease not in precautions['Disease'].values:
        print(f"\n No precautions found for: {disease}")
        return
    
    desc = description[description['Disease'] == disease]['Description'].values[0]


    # .iloc[:, 1:] skips the first column (which is usually 'Disease'), and selects all precaution columns (like Precaution_1, Precaution_2, etc.).
    # .values.flatten() converts them to a flat array/list.
    prec = precautions[precautions['Disease'] == disease].iloc[:, 1:].values.flatten()
    prec = [p for p in prec if str(p).lower() != 'nan']  # <- Add this line



    # Filters the medications DataFrame for the disease.
    # Extracts the 'Medication' column as a list of medications.
    meds = medications[medications['Disease'] == disease]['Medication'].tolist()
    if meds and isinstance(meds[0], str):  # <- Safely handle stringified list
        try:
            meds = ast.literal_eval(meds[0])
        except (ValueError, SyntaxError):
            meds = [meds[0]]  # fallback if it's just a plain string

    # Filters the diets DataFrame for the disease.
    # Extracts the recommended diet as a list.
    diet = diets[diets['Disease'] == disease]['Diet'].tolist()


    # Extracts the recommended workouts for the patient.
    work = workout[workout['disease'] == disease]['workout'].tolist()


    
    # Output Section
    print("\n--- Predicated Disease: ",disease)
    print("\n Description: ",desc)



    print("\n Precautions: ")
    for i, p in enumerate(prec, 1):
        print(f"{i}. {p}")

    
    print("\n Medications: ")
    for i,m in enumerate(meds,1):
        print(f"{i}. {m}")

    
    print("\nWorkout:")
    for i, w in enumerate(work, 1):
            print(f"{i}. {w}")



    print("\nDiet:")
    for i, d in enumerate(diet, 1):
        print(f"{i}. {d}")





In [68]:
# This line asks the user to type in symptoms, separated by commas.
user_input = input("Enter symptoms separted by commas: ")

# This splits the input string into a list of individual symptoms using , as the delimiter.
#If user_input = " fever, headache , cough "
# Then user_symptoms = ["fever", "headache", "cough"]

user_symptoms = [s.strip() for s in user_input.split(',')]


predicted = predict_disease(user_symptoms)
get_recommendation(predicted)



# Itching , Skin Rash , nodal skin eruptions
# stomach pain + acidity
# red spots,malaise, swelled lymph nodes


Input: red spots, Matched: red spots over body
Input: malaise, Matched: malaise
Input: swelled lymph nodes, Matched: swelled lymph nodes

--- Predicated Disease:  Chicken pox

 Description:  Chicken pox is a highly contagious viral infection causing an itchy rash.

 Precautions: 
1. Chicken pox
2. use neem in bathing 
3. consume neem leaves
4. take vaccine
5. avoid public places

 Medications: 
1. Antiviral drugs
2. Pain relievers
3. IV fluids
4. Blood transfusions
5. Platelet transfusions

Workout:
1. Stay hydrated
2. Include easily digestible foods
3. Include vitamin C-rich foods
4. Consume protein-rich foods
5. Include zinc-rich foods
6. Avoid spicy and acidic foods
7. Consult a healthcare professional
8. Practice good hygiene
9. Rest and conserve energy
10. Gradually resume normal diet

Diet:
1. ['Chicken Pox Diet', 'High-Calorie Diet', 'Soft and bland foods', 'Hydration', 'Protein-rich foods']


C:\Users\gaure\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
